In [ ]:
import pandas as pd
import numpy as np
import pickle
from google.colab import drive
from sklearn.preprocessing import LabelEncoder

# 1. Mount Google Drive and load the dataset
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/wheat_yield_india/dataset/final_usa_wheat_dataset.csv'
df = pd.read_csv(file_path)

# 2. Filter required columns and aggregate duplicates
features_and_target = [
    'Year', 'State', 'Dist Code', 'Yield', 'Yield_Lag1', 'Avg_Season_Tmax',
    'Total_Season_Rain', 'Sowing_Rainfall', 'Terminal_Heat_Tmax',
    'Temperature_Anomaly', 'Rainfall_Deviation'
]
df = df[features_and_target]
df = df.groupby(['Year', 'State', 'Dist Code'], as_index=False).mean().dropna()

# 3. Aggressive Feature Engineering to boost R2
# Weather Interactions
df['Temp_Rain_Ratio'] = df['Avg_Season_Tmax'] / (df['Total_Season_Rain'] + 1)
df['Heat_Stress_Index'] = df['Terminal_Heat_Tmax'] * df['Avg_Season_Tmax']
df['Rainfall_Impact'] = df['Total_Season_Rain'] * df['Rainfall_Deviation']

# Safe Historical District Yield (Expanding Mean shifted by 1 to prevent future data leakage)
df = df.sort_values(by=['Year', 'Dist Code'])
df['Historical_Dist_Yield'] = df.groupby('Dist Code')['Yield'].transform(lambda x: x.expanding().mean().shift(1))
df['Historical_Dist_Yield'] = df['Historical_Dist_Yield'].fillna(df['Yield_Lag1'])

# 4. Label Encode State and Dist Code, then save encoders for Streamlit
state_encoder = LabelEncoder()
dist_code_encoder = LabelEncoder()

df['State'] = state_encoder.fit_transform(df['State'])
df['Dist Code'] = dist_code_encoder.fit_transform(df['Dist Code'])

with open('state_encoder.pkl', 'wb') as f:
    pickle.dump(state_encoder, f)

with open('dist_code_encoder.pkl', 'wb') as f:
    pickle.dump(dist_code_encoder, f)

# 5. Cast to category dtype for native tree handling
df['State'] = df['State'].astype('category')
df['Dist Code'] = df['Dist Code'].astype('category')

# 6. Sort chronologically and split strictly by Future Years
df = df.sort_values(by=['Year', 'Dist Code']).reset_index(drop=True)

split_year = int(df['Year'].quantile(0.8))
train_data = df[df['Year'] <= split_year].copy()
test_data = df[df['Year'] > split_year].copy()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 3: Model Training
#### Part 1: Engineered Features with Extreme Gradient Boosting

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Define updated features (Now including 4 new engineered features)
features = [
    'Year', 'State', 'Dist Code', 'Yield_Lag1', 'Avg_Season_Tmax',
    'Total_Season_Rain', 'Sowing_Rainfall', 'Terminal_Heat_Tmax',
    'Temperature_Anomaly', 'Rainfall_Deviation',
    'Temp_Rain_Ratio', 'Heat_Stress_Index', 'Rainfall_Impact', 'Historical_Dist_Yield'
]
target = 'Yield'

# Fix whitespace in feature names
X_train = train_data[features].copy()
X_test = test_data[features].copy()
X_train.columns = X_train.columns.str.replace(' ', '_')
X_test.columns = X_test.columns.str.replace(' ', '_')

y_train = train_data[target]
y_test = test_data[target]

# 2. Initialize Models with parameters tuned for engineered features
models_part_13 = {
    "Engineered CatBoost": CatBoostRegressor(
        iterations=1500,
        learning_rate=0.03,
        depth=10,
        cat_features=['State', 'Dist_Code'],
        random_state=42,
        verbose=False
    ),
    "Engineered LightGBM": LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=12,
        num_leaves=63,
        min_child_samples=5,
        random_state=42,
        verbosity=-1
    ),
    "Engineered XGBoost": XGBRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=10,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        tree_method='hist',
        random_state=42,
        objective='reg:squarederror'
    )
}

# 3. Train and Evaluate
results_part_13 = []

for name, model in models_part_13.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    results_part_13.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    })

# 4. Print sorted leaderboard for Part 13
leaderboard_p13_df = pd.DataFrame(results_part_13).sort_values(by="R2", ascending=False).reset_index(drop=True)
print(leaderboard_p13_df)

                 Model        R2        RMSE         MAE
0   Engineered XGBoost  0.765279  808.028946  622.678508
1  Engineered CatBoost  0.762195  813.320771  617.969797
2  Engineered LightGBM  0.754664  826.098437  625.226834


#### Part 2: Corrected Mega Ensemble & RMSE Bug Fix

In [ ]:
# Install category_encoders if not already available
!pip install -q category_encoders

import numpy as np
import pandas as pd
from category_encoders import TargetEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import PowerTransformer
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Target Encoding (Makes features purely numeric)
cat_cols = ['State', 'Dist_Code']
X_train_te = X_train.copy()
X_test_te = X_test.copy()

# Ensure string types for clean Target Encoding
for col in cat_cols:
    if col in X_train_te.columns:
        X_train_te[col] = X_train_te[col].astype(str)
        X_test_te[col] = X_test_te[col].astype(str)

te = TargetEncoder(cols=cat_cols)
X_train_enc = te.fit_transform(X_train_te, y_train)
X_test_enc = te.transform(X_test_te)

# 2. Initialize Models and Target Transformers (Increased estimators and depth for max R2)
pt = PowerTransformer(method='yeo-johnson')

cat_base = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.015,
    depth=10,
    l2_leaf_reg=3,
    random_state=42,
    verbose=False
)

lgbm_base = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.015,
    max_depth=15,
    num_leaves=255,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

xgb_base = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.015,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    objective='reg:squarederror'
)

models_part_16 = {
    "Target-Encoded Transformed CatBoost": TransformedTargetRegressor(regressor=cat_base, transformer=pt),
    "Target-Encoded Transformed LightGBM": TransformedTargetRegressor(regressor=lgbm_base, transformer=pt),
    "Target-Encoded Transformed XGBoost": TransformedTargetRegressor(regressor=xgb_base, transformer=pt)
}

# 3. Create a Heavyweight Voting Ensemble
voting_ensemble = VotingRegressor(
    estimators=[
        ('te_cat', models_part_16["Target-Encoded Transformed CatBoost"]),
        ('te_lgbm', models_part_16["Target-Encoded Transformed LightGBM"]),
        ('te_xgb', models_part_16["Target-Encoded Transformed XGBoost"])
    ],
    n_jobs=-1
)
models_part_16["Mega Voting Ensemble (Cat+LGBM+XGB)"] = voting_ensemble

# 4. Train and Evaluate
results_part_16 = []

for name, model in models_part_16.items():
    model.fit(X_train_enc, y_train)
    y_pred = model.predict(X_test_enc)

    r2 = r2_score(y_test, y_pred)
    # FIX: Removed squared=False to resolve the TypeError in newer scikit-learn versions
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    results_part_16.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    })

# 5. Print sorted leaderboard for Part 16
leaderboard_p16_df = pd.DataFrame(results_part_16).sort_values(by="R2", ascending=False).reset_index(drop=True)
print(leaderboard_p16_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.7 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'catboost'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# pip install catboost scikit-learn pandas numpy xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# -----------------------------
# 1. Load Dataset & Split (USA)
# -----------------------------
# Ensure this path matches exactly where your USA moderate dataset is saved
df = pd.read_csv('/content/drive/MyDrive/wheat_yield_india/dataset/final_usa_wheat_dataset_moderate.csv')

train_df = df[df['Year'] <= 2013].copy()
test_df  = df[df['Year'] > 2013].copy()

target = 'Yield'
cat_features = ['State', 'Dist Code', 'District']
feature_cols = [c for c in df.columns if c not in [target, 'Country']]

y_train = train_df[target]
y_test  = test_df[target]

# -----------------------------
# 2. Prepare Data Formats
# -----------------------------
# Format A: One-Hot Encoded (For Scikit-Learn Models)
X_train_ohe = pd.get_dummies(train_df[feature_cols], columns=cat_features, drop_first=True)
X_test_ohe  = pd.get_dummies(test_df[feature_cols], columns=cat_features, drop_first=True)
X_train_ohe, X_test_ohe = X_train_ohe.align(X_test_ohe, join='left', axis=1, fill_value=0)

# Format B: Native Categorical (For XGBoost & CatBoost)
for col in cat_features:
    train_df[col] = train_df[col].astype('category')
    test_df[col]  = test_df[col].astype('category')

X_train_cat = train_df[feature_cols]
X_test_cat  = test_df[feature_cols]

# Array to store results for the leaderboard
results = []

def evaluate_model(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append({"Model": name, "R2 Score": r2, "RMSE": rmse, "MAE": mae})
    return model

# ======================================================
# 3. Train Scikit-Learn Models (Using OHE Data)
# ======================================================

print("Training Linear Models...")
evaluate_model("Linear Regression", LinearRegression(), X_train_ohe, X_test_ohe)
evaluate_model("Ridge Regression", Ridge(alpha=1.0), X_train_ohe, X_test_ohe)

print("Training Scikit-Learn Tree Models (Using OHE)...")
evaluate_model("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), X_train_ohe, X_test_ohe)
evaluate_model("Gradient Boosting", GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42), X_train_ohe, X_test_ohe)

# ======================================================
# 4. Train Advanced Models (Using Native Categories)
# ======================================================

print("Training XGBoost (Native Categories)...")
xgb_model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    tree_method='hist', enable_categorical=True
)
evaluate_model("XGBoost", xgb_model, X_train_cat, X_test_cat)

print("Training CatBoost (Native Categories)...")
cat_model = CatBoostRegressor(
    iterations=300, depth=6, learning_rate=0.1,
    loss_function='RMSE', random_seed=42, logging_level='Silent'
)
# CatBoost requires specific categorical feature indexing
cat_model.fit(X_train_cat, y_train, cat_features=cat_features)
preds_cat = cat_model.predict(X_test_cat)
results.append({
    "Model": "CatBoost",
    "R2 Score": r2_score(y_test, preds_cat),
    "RMSE": np.sqrt(mean_squared_error(y_test, preds_cat)),
    "MAE": mean_absolute_error(y_test, preds_cat)
})

# ======================================================
# 5. Display Leaderboard
# ======================================================
leaderboard = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False).reset_index(drop=True)
print("\n========================================")
print("   FINAL MODEL LEADERBOARD (USA)")
print("========================================")
display(leaderboard)



Training Linear Models...
Training Scikit-Learn Tree Models (Using OHE)...
Training XGBoost (Native Categories)...
Training CatBoost (Native Categories)...

   FINAL MODEL LEADERBOARD (USA)


,Model,R2 Score,RMSE,MAE
0,Random Forest,0.860917,596.322998,438.350528
1,CatBoost,0.858638,601.187318,456.678755
2,Ridge Regression,0.856386,605.958421,471.186483
3,Linear Regression,0.856212,606.324065,471.536607
4,XGBoost,0.844469,630.598249,479.121118
5,Gradient Boosting,0.843197,633.172300,473.079436


In [ ]:
import pickle
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# 1. Train the Random Forest model on the One-Hot Encoded data
rf_model_usa = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Note: X_train_ohe and y_train must already be in your notebook's memory
# from the previous benchmarking script.
rf_model_usa.fit(X_train_ohe, y_train)

# 2. Define Save Paths
model_save_path = '/content/drive/MyDrive/wheat_yield_india/dataset/usa_rf_model.pkl'
metadata_save_path = '/content/drive/MyDrive/wheat_yield_india/dataset/usa_rf_metadata.pkl'

# 3. Save the Model
with open(model_save_path, 'wb') as f:
    pickle.dump(rf_model_usa, f)

# 4. Save the Metadata (CRITICAL: Must save the OHE columns)
ohe_features = list(X_train_ohe.columns)

metadata = {
    'features': ohe_features, # The frontend will use this to align the columns
    'cat_features': cat_features,
    'target': target
}

with open(metadata_save_path, 'wb') as f:
    pickle.dump(metadata, f)

print("USA Random Forest model and OHE metadata exported successfully!")

USA Random Forest model and OHE metadata exported successfully!
